# 🏥 Notebook 2 — Skills: Making Claire Smart About Different Tasks

In Notebook 1 you built a working agent. But it treats every question the same.  
**Skills** are modular instructions that tell the agent *how* to handle specific tasks:

- "Show me the PA dashboard" → runs a specific sequence of queries
- "Review this PA against care pathways" → runs gap analysis
- "Check for fraud" → queries the alerts table

You'll build skills, test them against live Teradata data, and see how  
the same agent handles completely different workflows.

In [ ]:
!pip install --quiet strands-agents mcp boto3 python-dotenv bedrock-agentcore 2>/dev/null
# ── Setup (reuse from Notebook 1) ──
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["PATH"] = f"{os.path.expanduser('~')}/.local/bin:{os.environ['PATH']}"

from strands import Agent
from strands.models.bedrock import BedrockModel
from strands.tools.mcp import MCPClient
from mcp import stdio_client, StdioServerParameters

model = BedrockModel(model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0", streaming=False)

server_params = StdioServerParameters(
    command="uvx",
    args=["teradata-mcp-server"],
    env={
        "DATABASE_URI": os.getenv("TERADATA_DATABASE_URI"),
        "LOGMECH": os.getenv("LOGMECH", "TD2")
    }
)
teradata_tool = MCPClient(lambda: stdio_client(server_params), startup_timeout=300)

print("Model + Teradata MCP ready.")

---
## 2.1 — Define Skills

A **skill** is a block of instructions injected into the system prompt  
based on what the user asks. The agent gets different "playbooks" for different tasks.

In [ ]:
# ── Base prompt (always included) ──
BASE_PROMPT = """You are Claire, an intelligent healthcare claims and prior authorization AI agent.
You have access to a Teradata HCLS database via MCP tools.

Key tables: prior_auth_request, member, provider, facility, auth_cpt_code, cpt_codes,
auth_icd_code, auth_clinical_notes, auth_status, care_pathways, auth_request_detail,
treatment_type, prior_treatment_duration, WORKERS_COMP_HDR, WORKERS_COMP_DTL, alerts.

CRITICAL: Use ordering_provider_id (NOT provider_id). Use primary_specialty (NOT specialty).
Use Teradata SQL (TOP instead of LIMIT). alerts table uses object_id for provider NPI.
"""

# ── Skills ──
SKILLS = {
    "pa-dashboard": {
        "trigger_words": ["dashboard", "display", "show", "view", "details", "summary"],
        "prompt": """
## PA Dashboard Skill
When asked to show/display a PA dashboard, follow these steps:

1. Query prior_auth_request WHERE auth_request_num = '{pa_number}'
   Save auth_request_id, member_id, ordering_provider_id, facility_id, auth_status_id.

2. Query member, provider, facility, auth_status using the IDs from step 1.

3. Query auth_cpt_code (joined with cpt_codes), auth_icd_code, auth_clinical_notes.

Present ALL data in a structured summary. Do NOT analyze care pathways.
"""
    },
    "care-pathway-review": {
        "trigger_words": ["review", "pathway", "prerequisites", "gap", "analyze", "care pathway"],
        "prompt": """
## Care Pathway Review Skill
When asked to review a PA against care pathways:

1. Get the PA header and member_id_num from prior_auth_request + member.
2. Get CPT codes from auth_cpt_code.
3. Check care_pathways table for prerequisites: 
   SELECT cp.cpt_code, cp.cpt_code_prereq, c.cpt_desc
   FROM HCLS.care_pathways cp JOIN HCLS.cpt_codes c ON cp.cpt_code_prereq = c.cpt_code
   WHERE cp.cpt_code IN (requested CPT codes)

4. For each prerequisite, check claims history:
   SELECT h.Bill_ID, h.Service_Bill_From_Date, d.HCPCS_Line_Procedure_Billed_Code
   FROM HCLS.WORKERS_COMP_HDR h JOIN HCLS.WORKERS_COMP_DTL d ON h.Bill_ID = d.Bill_ID
   WHERE h.Patient_Account_Number = '{member_id_num}'
   AND d.HCPCS_Line_Procedure_Billed_Code = '{prereq_cpt}'

5. Color code: GREEN (found in claims), AMBER (on PA but no claim), RED (missing)
6. Check fraud alerts: SELECT * FROM HCLS.alerts WHERE object_id = '{provider_npi}'
"""
    },
    "fraud-check": {
        "trigger_words": ["fraud", "alert", "integrity", "suspicious"],
        "prompt": """
## Fraud Check Skill
When asked to check fraud alerts:
1. Get the provider NPI from the PA (join prior_auth_request.ordering_provider_id to provider.npi)
2. Query: SELECT * FROM HCLS.alerts WHERE object_id = '{provider_npi}'
3. Report alert_type, alert_score, alert_priority, alert_description, alert_status
4. If no alerts, report the provider has no fraud alerts on file
"""
    }
}

def match_skill(user_message):
    """Match user message to best skill."""
    msg = user_message.lower()
    best, best_score = None, 0
    for name, skill in SKILLS.items():
        hits = sum(1 for w in skill["trigger_words"] if w in msg)
        if hits > best_score:
            best, best_score = name, hits
    return (best, SKILLS[best]["prompt"]) if best and best_score >= 1 else (None, "")

# Test matching
for msg in ["Show me the dashboard for PA-2026-003417",
            "Run the care pathway review for PA-2026-003417",
            "Check fraud alerts for the provider",
            "What is a prior auth?"]:
    name, _ = match_skill(msg)
    print(f"  '{msg[:50]}' → {name or '(no skill)'}")

### How would this work in production?

The keyword matcher above works for a demo, but production systems use smarter routing:

**Level 1 — Keyword match** (what we just built)
Simple, fast, zero cost. Good for 5-10 well-defined skills. Breaks when users phrase things unexpectedly.

**Level 2 — LLM-based classifier**
Use a fast, cheap model (Haiku) as a router. Feed it the user message + a list of skill names/descriptions, have it pick the best one. Costs ~0.1 cents per classification but handles any phrasing.

```python
# Example: LLM router
router_prompt = f"""Given these skills:
- pa-dashboard: Display PA details and status
- care-pathway-review: Check CPT prerequisites against claims history
- fraud-check: Check provider fraud alerts

User said: \"{user_message}\"
Which skill best matches? Reply with just the skill name, or 'none'."""

# One fast Bedrock call → get the skill name → inject the right prompt
```

**Level 3 — Embedding similarity**
Pre-compute embeddings for each skill description. At runtime, embed the user message and find the closest skill by cosine similarity. Fast, no LLM call needed, scales to hundreds of skills.

**Level 4 — Let the agent decide (tool-based)**
Don't route at all — give the agent ALL skills in the system prompt and let it pick which workflow to follow. This is what Claire's production app does. Works well when the model is strong enough, but uses more tokens.

**Level 5 — AgentCore managed harness**
Flatten all skills into the system prompt and let AgentCore handle everything. The harness manages the agent loop, and you iterate on the prompt. This is what we'll do in Notebook 3.

For this workshop, keyword matching keeps things transparent — you can see exactly which skill fires and why. In production, Level 2 or Level 4 are most common.

---
## 2.2 — Agent with Skill Routing

The right skill gets injected into the prompt *before* the agent sees the question.

In [ ]:
def run_claire(question):
    """Run Claire with automatic skill routing."""
    skill_name, skill_prompt = match_skill(question)
    full_prompt = BASE_PROMPT + skill_prompt
    
    if skill_name:
        print(f"🎯 Skill matched: {skill_name}")
    else:
        print(f"💬 No skill — general response")
    
    agent = Agent(model=model, tools=[teradata_tool], system_prompt=full_prompt)
    result = agent(question)
    return result.message

print("Claire with skills is ready!")

---
## 2.3 — Test: PA Dashboard (Live Data)

In [ ]:
response = run_claire("Show me the dashboard for PA-2026-003417")
print(response)

---
## 2.4 — Test: Care Pathway Review (Live Data)

In [ ]:
response = run_claire("Run the care pathway review for PA-2026-003417")
print(response)

---
## 2.5 — Test: Fraud Check (Live Data)

In [ ]:
response = run_claire("Check fraud alerts for the provider on PA-2026-003417")
print(response)

---
## 2.6 — Create Your Own Skill

Try adding a new skill! Copy a skill definition above and modify it.  
For example, a **member lookup** skill:

In [ ]:
# ── Add your own skill ──
SKILLS["member-lookup"] = {
    "trigger_words": ["member", "patient", "lookup", "find member"],
    "prompt": """
## Member Lookup Skill
When asked to look up a member:
1. Search the member table by name, member_id_num, or date_of_birth
2. Show all PAs for that member from prior_auth_request
3. Show their claims history from WORKERS_COMP_HDR
4. Summarize: total PAs, statuses, total claims, date range
"""
}

# Test it
response = run_claire("Look up member MBR-884201 and show me all their prior authorizations")
print(response)

---
## 2.7 — The Production Architecture

What you just built is the **core** of the Claire production app:

```
Claire Production Stack              What You Built Here
─────────────────────                ─────────────────────
React Frontend                       (notebook cell output)
FastAPI + SSE streaming (300 lines)  → Strands SDK (3 lines)
MCP subprocess manager (130 lines)   → MCPClient (5 lines)  
Skills loader (100 lines)            → match_skill() (10 lines)
Auth + sessions (80 lines)           → IAM role (0 lines)
```

In Notebook 3, we deploy this to **AgentCore** and delete even the Strands code.

---

**➡ Continue to [Notebook 3 — Deploy to AgentCore](03-agentcore-deploy.ipynb)**